# 地产去库存预测 重构

本项目基于EDB数据库中的房地产相关时间序列数据，对住宅库存进行预测分析。

## 1. 项目概述

### 1.1 功能描述

本项目提供**两种预测方法**：
1. **趋势推演预测**：基于历史趋势的线性回归
2. **销售-竣工去化预测**：基于供需平衡的动态预测

### 1.2 业务背景
- 跟踪中国房地产市场住宅库存变化
- 预测未来库存水平（预测到2027年12月）
- 计算库销比指标，评估市场去化压力

### 1.3 数据源（EDB数据库指标）

| 代码 | 指标名称 | 数据频率 |
|------|----------|----------|
| S0029674 | 中国:商品房待售面积:住宅:累计值 | 月度 |
| S0049264 | 商品房销售面积:住宅:现房:累计值 | 月度 |
| S0049296 | 商品房销售面积:住宅:期房:累计值 | 月度 |
| S0049585 | 中国:房屋竣工面积:住宅:累计值 | 月度 |
| S0029669 | 中国:房屋新开工面积:住宅:累计值 | 月度 |

### 1.4 输出结果

- 住宅库存预测（两种口径）
- 库销比指标（两种口径）
- 12个月滚动销售/竣工/新开工数据

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import datetime
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

print('依赖导入成功')

### 2.2 配置参数（数据代码映射）

In [ ]:
# EDB数据代码映射
TRADE_CODE_MAPPING = {
    'S0029674': '中国:商品房待售面积:住宅:累计值',
    'S0049264': '商品房销售面积:住宅:现房:累计值', 
    'S0049296': '商品房销售面积:住宅:期房:累计值',
    'S0049585': '中国:房屋竣工面积:住宅:累计值',
    'S0029669': '中国:房屋新开工面积:住宅:累计值'
}

# 数据代码列表
TRADE_CODES = list(TRADE_CODE_MAPPING.keys())

# 预测参数
FORECAST_CONFIG = {
    'start_date': '2024-10-01',  # 趋势分析起始日期
    'end_date': '2027-12-31',    # 预测结束日期
    'rolling_window': 12,        # 滚动窗口（月）
    'min_train_points': 6        # 最小训练数据点数
}

# 数据库配置（从环境变量获取）
DB_CONFIG = {
    'user': os.environ.get('DB_USER', 'hz_work'),
    'password': os.environ.get('DB_PASSWORD', ''),
    'host': os.environ.get('DB_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com'),
    'port': os.environ.get('DB_PORT', '3306'),
    'database': os.environ.get('DB_DATABASE', 'edb'),
    'charset': 'utf8mb4'
}

# 输出配置
OUTPUT_CONFIG = {
    'excel_filename': 'real_estate_destocking_forecast.xlsx',
    'plot_filename': 'real_estate_destocking_forecast.png',
    'plot_dpi': 300,
    'figure_size': (15, 12)
}

print('配置参数加载完成')

## 3. 数据获取

### 3.1 数据库连接

In [ ]:
def get_connection_url(db_config, db_type='mysql')
    """
    获取数据库连接URL
    
    Args:
        db_config: 数据库配置字典
        db_type: 数据库类型，默认为mysql
    
    Returns:
        str: SQLAlchemy连接URL
    """
    if db_type == 'mysql':
        driver = 'pymysql'
        return f"mysql+{driver}://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['database']}?charset={db_config['charset']}"
    elif db_type == 'postgresql':
        driver = 'psycopg2'
        return f"postgresql+{driver}://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['database']}"
    else:
        raise ValueError(f"不支持的数据库类型: {db_type}")

print('数据库连接函数定义完成')

### 3.2 EDB数据读取

In [ ]:
def get_edb_data(trade_codes, db_config=None):
    """
    获取edb数据库中的时间序列数据
    
    Args:
        trade_codes: 数据代码列表
        db_config: 数据库配置，默认使用全局DB_CONFIG
    
    Returns:
        DataFrame: 宽表格式的时序数据
    """
    if db_config is None:
        db_config = DB_CONFIG
    
    engine = create_engine(get_connection_url(db_config, 'mysql'))
    
    # 将trade_codes列表转换为占位符
    placeholders = ','.join(['%s'] * len(trade_codes))
    query = f"SELECT dt, trade_code, close FROM edb.edbdata WHERE trade_code IN ({placeholders}) ORDER BY dt"
    
    print("正在从edb数据库获取数据...")
    df = pd.read_sql(query, engine, params=tuple(trade_codes))
    df['dt'] = pd.to_datetime(df['dt'])
    
    # 转换为宽表格式
    pivot_df = df.pivot(index='dt', columns='trade_code', values='close')
    print(f"获取数据成功，数据范围：{pivot_df.index.min()} 到 {pivot_df.index.max()}")
    
    return pivot_df

print('EDB数据读取函数定义完成')

## 4. 数据处理

### 4.1 月度值计算

In [ ]:
def calculate_monthly_values(df):
    """
    计算月度值和累计值
    
    将年度累计值转换为月度值，并计算历史累计值
    
    Args:
        df: 原始数据DataFrame
    
    Returns:
        DataFrame: 包含月度值和历史累计值
    """
    print("计算月度值和历史累计值...")
    
    df_monthly = pd.DataFrame(index=df.index)
    
    # 需要转换的列
    columns_to_convert = ['S0049264', 'S0049296', 'S0049585', 'S0029669']
    
    for col in columns_to_convert:
        if col not in df.columns:
            continue
            
        df_reset = df[col].copy()
        
        # 检测年初重置点（当前值小于前一个值时）
        # 注意：可能存在1月份数据断开的情况，1月可能不公布数据，2月集中公布1-2月的累计值
        diff_values = df_reset.diff()
        reset_points = diff_values < 0
        
        # 计算当月值
        monthly_values = diff_values.fillna(df_reset)
        # 在重置点用原值作为当月值
        monthly_values[reset_points] = df_reset[reset_points]
        
        df_monthly[f'{col}_monthly'] = monthly_values
    
    # 计算历史累计值
    df_monthly['S0049264_hist_cum'] = df_monthly['S0049264_monthly'].cumsum()
    df_monthly['S0049296_hist_cum'] = df_monthly['S0049296_monthly'].cumsum()
    
    return df_monthly

print('月度值计算函数定义完成')

### 4.2 滚动指标计算（12个月）

In [ ]:
def calculate_rolling_indicators(df_monthly, window=12):
    """
    计算12个月滚动指标
    
    Args:
        df_monthly: 月度数据DataFrame
        window: 滚动窗口大小，默认12个月
    
    Returns:
        DataFrame: 包含滚动指标
    """
    print("计算12个月滚动指标...")
    
    rolling_data = pd.DataFrame(index=df_monthly.index)
    
    # 销售12个月滚动
    rolling_data['s1'] = df_monthly['S0049264_monthly'].rolling(window=window, min_periods=1).sum()  # 现房销售
    rolling_data['s2'] = df_monthly['S0049296_monthly'].rolling(window=window, min_periods=1).sum()  # 期房销售
    rolling_data['s3'] = rolling_data['s1'] + rolling_data['s2']  # 总销售
    
    # 竣工12个月滚动
    rolling_data['completion_12m'] = df_monthly['S0049585_monthly'].rolling(window=window, min_periods=1).sum()
    
    # 新开工12个月滚动
    rolling_data['new_starts_12m'] = df_monthly['S0029669_monthly'].rolling(window=window, min_periods=1).sum()
    
    return rolling_data

print('滚动指标计算函数定义完成')

## 5. 核心逻辑

### 5.1 住宅库存估算

In [ ]:
def calculate_inventory_estimate(df, df_monthly):
    """
    计算住宅库存估算值
    
    公式: 住宅库存估算值 = 待售面积 * (期房累计 + 现房累计) / 期房累计
    
    Args:
        df: 原始数据DataFrame
        df_monthly: 月度数据DataFrame
    
    Returns:
        Series: 住宅库存估算值
    """
    print("计算住宅库存估算值...")
    
    # 住宅库存估算值 = S0029674 * (期房历史累计 + 现房历史累计) / 期房历史累计
    inventory_estimate = df['S0029674'] * \
        (df_monthly['S0049296_hist_cum'] + df_monthly['S0049264_hist_cum']) / \
        df_monthly['S0049296_hist_cum']
    
    return inventory_estimate

print('住宅库存估算函数定义完成')

### 5.2 趋势线性回归预测

In [ ]:
def trend_forecast(data, forecast_start_date, forecast_end_date):
    """
    趋势线性回归预测
    
    基于历史数据的线性回归趋势进行未来值预测
    
    Args:
        data: 时序数据DataFrame
        forecast_start_date: 趋势分析起始日期
        forecast_end_date: 预测结束日期
    
    Returns:
        DataFrame: 预测结果
    """
    print("执行趋势线性回归预测...")
    
    # 将预测开始日期转换为datetime类型以确保正确的比较
    forecast_start_dt = pd.to_datetime(forecast_start_date)
    
    # 准备训练数据（指定日期及之后）
    train_data = data[data.index >= forecast_start_dt].dropna()
    
    if len(train_data) < 2:
        # 如果指定日期后的数据不足，尝试使用最近的24个月数据
        print("指定日期后的训练数据点不足，尝试使用最近的24个月数据...")
        train_data = data.tail(24).dropna()
        
        if len(train_data) < 2:
            raise ValueError(f"训练数据点不足，无法进行预测。数据范围: {data.index.min()} 到 {data.index.max()}")
    
    # 创建时间变量
    train_data = train_data.copy()
    train_data['time'] = range(len(train_data))
    
    # 创建预测日期范围
    last_date = data.index.max()
    future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), 
                                end=forecast_end_date, freq='M')
    
    # 线性回归模型
    model = LinearRegression()
    X_train = train_data[['time']]
    y_train = train_data.iloc[:, 0]  # 第一列数据
    
    model.fit(X_train, y_train)
    
    # 预测
    future_time = range(len(train_data), len(train_data) + len(future_dates))
    future_values = model.predict(np.array(future_time).reshape(-1, 1))
    
    # 创建预测结果
    forecast_df = pd.DataFrame(index=future_dates, data=future_values, columns=[data.columns[0]])
    
    return forecast_df

print('趋势预测函数定义完成')

### 5.3 销售-竣工去化预测

In [ ]:
def sales_completion_destocking_forecast(inventory_estimate, rolling_data, forecast_start_date, forecast_end_date):
    """
    销售-竣工去化预测
    
    基于供需平衡的动态预测：库存变化 = 销售 - 竣工
    
    Args:
        inventory_estimate: 库存估算值Series
        rolling_data: 滚动指标DataFrame
        forecast_start_date: 趋势分析起始日期
        forecast_end_date: 预测结束日期
    
    Returns:
        Series: 库存预测结果
    """
    print("执行销售-竣工去化预测...")
    
    # 获取销售和竣工的趋势预测
    s3_forecast = trend_forecast(rolling_data[['s3']], forecast_start_date, forecast_end_date)
    completion_forecast = trend_forecast(rolling_data[['completion_12m']], forecast_start_date, forecast_end_date)
    
    # 计算月度销售和竣工
    monthly_sales = s3_forecast['s3'] / 12
    monthly_completion = completion_forecast['completion_12m'] / 12
    
    # 计算月度去化值（销售 - 竣工）
    monthly_destocking = monthly_sales - monthly_completion
    
    # 从最后实际库存开始，逐月计算
    last_inventory = inventory_estimate.dropna().iloc[-1]
    inventory_forecast = []
    
    for destocking in monthly_destocking:
        if len(inventory_forecast) == 0:
            new_inventory = last_inventory - destocking
        else:
            new_inventory = inventory_forecast[-1] - destocking
        inventory_forecast.append(new_inventory)
    
    return pd.Series(inventory_forecast, index=monthly_destocking.index)

print('去化预测函数定义完成')

### 5.4 库销比计算

In [ ]:
def calculate_inventory_sales_ratio(final_df):
    """
    计算库销比
    
    公式: 库销比 = 库存 / (年度销售/12)
    
    Args:
        final_df: 包含库存和销售数据的DataFrame
    
    Returns:
        DataFrame: 添加库销比列后的DataFrame
    """
    print("计算库销比...")
    
    # 计算库销比
    final_df['库销比（趋势推演）'] = final_df['住宅库存趋势预测口径'] / (final_df['住宅销售12个月滚动'] / 12)
    final_df['库销比（销售-竣工去化）'] = final_df['住宅库存去化预测口径'] / (final_df['住宅销售12个月滚动'] / 12)
    
    return final_df

print('库销比计算函数定义完成')

### 5.5 创建最终结果DataFrame

In [ ]:
def create_final_dataframe(df, rolling_data, inventory_estimate, forecast_start_date, forecast_end_date):
    """
    创建最终结果DataFrame
    
    整合历史数据和预测数据，计算库销比
    
    Args:
        df: 原始数据DataFrame
        rolling_data: 滚动指标DataFrame
        inventory_estimate: 库存估算值Series
        forecast_start_date: 趋势分析起始日期
        forecast_end_date: 预测结束日期
    
    Returns:
        DataFrame: 最终结果
    """
    print("创建最终结果DataFrame...")
    
    # 1. 趋势预测
    inventory_trend_forecast = trend_forecast(
        inventory_estimate.to_frame('inventory'), forecast_start_date, forecast_end_date
    )
    s3_trend_forecast = trend_forecast(rolling_data[['s3']], forecast_start_date, forecast_end_date)
    completion_trend_forecast = trend_forecast(rolling_data[['completion_12m']], forecast_start_date, forecast_end_date)
    new_starts_trend_forecast = trend_forecast(rolling_data[['new_starts_12m']], forecast_start_date, forecast_end_date)
    
    # 2. 销售-竣工去化预测
    inventory_destocking_forecast = sales_completion_destocking_forecast(
        inventory_estimate, rolling_data, forecast_start_date, forecast_end_date
    )
    
    # 3. 合并历史和预测数据
    all_dates = pd.date_range(start=df.index.min(), end=forecast_end_date, freq='M')
    final_df = pd.DataFrame(index=all_dates)
    
    # 住宅库存趋势预测口径
    final_df['住宅库存趋势预测口径'] = pd.concat([
        inventory_estimate, 
        inventory_trend_forecast['inventory']
    ]).reindex(all_dates)
    
    # 住宅库存去化预测口径
    final_df['住宅库存去化预测口径'] = pd.concat([
        inventory_estimate,
        inventory_destocking_forecast
    ]).reindex(all_dates)
    
    # 住宅销售12个月滚动
    final_df['住宅销售12个月滚动'] = pd.concat([
        rolling_data['s3'],
        s3_trend_forecast['s3']
    ]).reindex(all_dates)
    
    # 住宅竣工12个月滚动
    final_df['住宅竣工12个月滚动'] = pd.concat([
        rolling_data['completion_12m'],
        completion_trend_forecast['completion_12m']
    ]).reindex(all_dates)
    
    # 住宅新开工12个月滚动
    final_df['住宅新开工12个月滚动'] = pd.concat([
        rolling_data['new_starts_12m'],
        new_starts_trend_forecast['new_starts_12m']
    ]).reindex(all_dates)
    
    # 计算库销比
    final_df = calculate_inventory_sales_ratio(final_df)
    
    return final_df

print('最终结果DataFrame函数定义完成')

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
# 计算预测参数
current_date = datetime.date.today()
forecast_start_date = (current_date - datetime.timedelta(days=3*365)).strftime('%Y-%m-%d')
forecast_end_date = FORECAST_CONFIG['end_date']

print(f"趋势分析起始日期: {forecast_start_date}")
print(f"预测结束日期: {forecast_end_date}")

In [ ]:
# 1. 获取数据
df = get_edb_data(TRADE_CODES)
print(f"数据维度: {df.shape}")
df.head()

In [ ]:
# 2. 计算月度值
df_monthly = calculate_monthly_values(df)
df_monthly.head()

In [ ]:
# 3. 计算库存估算值
inventory_estimate = calculate_inventory_estimate(df, df_monthly)
inventory_estimate.tail()

In [ ]:
# 4. 计算滚动指标
rolling_data = calculate_rolling_indicators(df_monthly, window=FORECAST_CONFIG['rolling_window'])
rolling_data.tail()

In [ ]:
# 5. 创建最终结果
final_df = create_final_dataframe(
    df, rolling_data, inventory_estimate, 
    forecast_start_date, forecast_end_date
)
final_df.tail(20)

### 6.2 结果验证

In [ ]:
# 验证数据完整性
print("=== 数据完整性检查 ===")
print(f"总行数: {len(final_df)}")
print(f"日期范围: {final_df.index.min()} 到 {final_df.index.max()}")
print("\n各列非空值数量:")
print(final_df.count())
print("\n各列统计信息:")
final_df.describe()

In [ ]:
# 保存结果到Excel
output_path = 'output/' + OUTPUT_CONFIG['excel_filename']
final_df.to_excel(output_path)
print(f"结果已保存到: {output_path}")

## 7. 工具函数（可复用）

In [ ]:
def plot_forecast_results(final_df, output_path=None, figsize=(15, 12)):
    """
    绘制预测结果图表
    
    Args:
        final_df: 结果DataFrame
        output_path: 图片保存路径，如果为None则不保存
        figsize: 图表大小
    """
    fig, axes = plt.subplots(3, 1, figsize=figsize)
    
    # 库存预测
    ax1 = axes[0]
    final_df['住宅库存趋势预测口径'].plot(ax=ax1, label='趋势预测口径')
    final_df['住宅库存去化预测口径'].plot(ax=ax1, label='去化预测口径')
    ax1.axvline(x=pd.Timestamp.now(), color='r', linestyle='--', label='当前')
    ax1.set_title('住宅库存预测')
    ax1.legend()
    ax1.grid(True)
    
    # 库销比
    ax2 = axes[1]
    final_df['库销比（趋势推演）'].plot(ax=ax2, label='趋势推演库销比')
    final_df['库销比（销售-竣工去化）'].plot(ax=ax2, label='去化预测库销比')
    ax2.axvline(x=pd.Timestamp.now(), color='r', linestyle='--', label='当前')
    ax2.set_title('库销比')
    ax2.legend()
    ax2.grid(True)
    
    # 滚动指标
    ax3 = axes[2]
    final_df['住宅销售12个月滚动'].plot(ax=ax3, label='销售12M滚动')
    final_df['住宅竣工12个月滚动'].plot(ax=ax3, label='竣工12M滚动')
    final_df['住宅新开工12个月滚动'].plot(ax=ax3, label='新开工12M滚动')
    ax3.axvline(x=pd.Timestamp.now(), color='r', linestyle='--', label='当前')
    ax3.set_title('12个月滚动指标')
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存到: {output_path}")
    
    plt.show()

print('绘图函数定义完成')

In [ ]:
# 绘制结果图表
plot_output_path = 'assets/' + OUTPUT_CONFIG['plot_filename']
plot_forecast_results(final_df, plot_output_path)